# evaluatorB â€” Primary temporal-localization scorer (ID-agnostic tIoU)

Primary temporal-localization scorer (ID-agnostic tIoU). Run all cells top-to-bottom.

**Protocol** (ActivityNet-style temporal matching, single interaction class):
1. Intervals are inclusive frame spans `[start, end]`.
2. Pairwise temporal IoU over the full GT Ã— pred matrix (IDs ignored):
   `inter = max(0, min(e1,e2) âˆ’ max(s1,s2) + 1)`, `union = dur_GT + dur_pred âˆ’ inter`.
3. One-to-one assignment via Hungarian algorithm on `âˆ’tIoU`; zero-overlap pairs discarded.
4. Per threshold `t`: `TP` = matched pairs with `tIoU â‰¥ t`, `FP = n_pred âˆ’ TP`, `FN = n_gt âˆ’ TP` â†’ standard P/R/F1.
   `mean_F1` = mean of F1 over thresholds (NOT mAP â€” the pipeline emits binary intervals with no confidence scores).

Pred summaries carry the pipeline's frame stitching (threshold=60, min-length=30, stride-5); GT does not, so ~1-stride boundary offsets are inherent.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
pd.options.display.float_format = "{:.2f}".format

CWD = Path.cwd()
if (CWD / "eval" / "gt_summary").exists():
    ROOT = CWD            # kernel launched from src/
elif (CWD / "gt_summary").exists():
    ROOT = CWD.parent     # kernel launched from src/eval/
else:
    raise SystemExit(f"Cannot locate gt_summary from {CWD}")

GT_DIR = ROOT / "eval" / "gt_summary"
PRED_DIR = ROOT / "video" / "logs" / "logs_summary"
OUT_CSV = ROOT / "eval" / "result" / "primary.csv"
THRESHOLDS = (0.3, 0.5, 0.75)
GT_PATTERN = "vid*_true_summary.csv"
PRED_PATTERN = "vid*_summary_log.csv"

print(f"GT:   {GT_DIR}")
print(f"PRED: {PRED_DIR}")
print(f"OUT:  {OUT_CSV}")

In [ ]:
def _vidnum(name):
    m = re.search(r"vid(\d+)", name)
    return int(m.group(1)) if m else None


def pair_tiou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]) + 1)
    union = (a[1] - a[0] + 1) + (b[1] - b[0] + 1) - inter
    return inter / union if union > 0 else 0.0


def score_video(gt_path, pred_path, thresholds=THRESHOLDS):
    g = pd.read_csv(gt_path)[["frame_start", "frame_end"]].values.tolist()
    p = pd.read_csv(pred_path)[["frame_start", "frame_end"]].values.tolist()
    n_gt, n_pred = len(g), len(p)
    out = {"n_gt": n_gt, "n_pred": n_pred}
    if n_gt == 0 or n_pred == 0:
        for t in thresholds:
            out[f"P@{t:g}"], out[f"R@{t:g}"], out[f"F1@{t:g}"] = 0.0, 0.0, 0.0
        out["mean_F1"], out["n_match"] = 0.0, 0
        out["tp_at"] = {t: 0 for t in thresholds}
        return out
    M = np.array([[pair_tiou(a, b) for b in p] for a in g])
    row, col = linear_sum_assignment(-M)
    matched = sorted([float(M[r, c]) for r, c in zip(row.tolist(), col.tolist()) if M[r, c] > 0], reverse=True)
    out["n_match"] = len(matched)
    f1s = []
    for t in thresholds:
        tp = sum(1 for v in matched if v >= t)
        P = tp / n_pred if n_pred else 0.0
        R = tp / n_gt if n_gt else 0.0
        F = 2 * P * R / (P + R) if (P + R) else 0.0
        out[f"P@{t:g}"], out[f"R@{t:g}"], out[f"F1@{t:g}"] = P, R, F
        f1s.append(F)
    out["mean_F1"] = float(np.mean(f1s))
    out["tp_at"] = {t: sum(1 for v in matched if v >= t) for t in thresholds}
    return out


def discover(gt_dir, pred_dir):
    gt = {n: f for f in gt_dir.glob(GT_PATTERN) if (n := _vidnum(f.name)) is not None}
    pr = {n: f for f in pred_dir.glob(PRED_PATTERN) if (n := _vidnum(f.name)) is not None}
    pairs = [(v, gt[v], pr[v]) for v in sorted(set(gt) & set(pr))]
    for v in sorted(set(gt) - set(pr)):
        print(f"skip vid{v:02d}: GT without prediction ({gt[v].name})")
    for v in sorted(set(pr) - set(gt)):
        print(f"skip vid{v:02d}: prediction without GT ({pr[v].name})")
    return pairs

In [ ]:
pairs = discover(GT_DIR, PRED_DIR)
print(f"scoring {len(pairs)} videos")

rows = []
for v, gp, pp in pairs:
    rows.append({"video": f"vid{v:02d}", **score_video(gp, pp, THRESHOLDS)})
df = pd.DataFrame(rows)

cols = ["video", "n_gt", "n_pred"] + [c for t in THRESHOLDS for c in (f"P@{t:g}", f"R@{t:g}", f"F1@{t:g}")] + ["mean_F1", "n_match"]
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
df[cols]

In [ ]:
n_gt_tot = int(df["n_gt"].sum())
n_pred_tot = int(df["n_pred"].sum())
micro = {}
for t in THRESHOLDS:
    tp = int(sum(d.get(t, 0) for d in df["tp_at"]))
    P = tp / n_pred_tot if n_pred_tot else 0.0
    R = tp / n_gt_tot if n_gt_tot else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    micro[f"P@{t:g}"], micro[f"R@{t:g}"], micro[f"F1@{t:g}"] = P, R, F
micro["mean_F1"] = float(np.mean([micro[f"F1@{t:g}"] for t in THRESHOLDS]))

print(f"=== MICRO (pooled over interactions, HEADLINE, n_gt={n_gt_tot}, n_pred={n_pred_tot}) ===")
for t in THRESHOLDS:
    print(f"t={t:g}  P {micro[f'P@{t:g}']:.2f}  R {micro[f'R@{t:g}']:.2f}  F1 {micro[f'F1@{t:g}']:.2f}")
print(f"mean_F1 {micro['mean_F1']:.2f}")
print()
print(f"=== MACRO (mean +- sample std over videos, n={len(df)}) ===")
for c in cols[3:]:
    print(f"{c:10s} {df[c].mean():.2f} +- {df[c].std(ddof=1):.2f}")

SUMMARY_CSV = ROOT / "eval" / "result" / "primary_summary.csv"
srows = [{"level": "micro", **micro, "n_gt": n_gt_tot, "n_pred": n_pred_tot, "n_videos": len(df)},
         {"level": "macro",
          **{f"P@{t:g}": float(df[f"P@{t:g}"].mean()) for t in THRESHOLDS},
          **{f"R@{t:g}": float(df[f"R@{t:g}"].mean()) for t in THRESHOLDS},
          **{f"F1@{t:g}": float(df[f"F1@{t:g}"].mean()) for t in THRESHOLDS},
          "mean_F1": float(df["mean_F1"].mean()),
          "n_gt": n_gt_tot, "n_pred": n_pred_tot, "n_videos": len(df)}]
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.drop(columns=["tp_at"]).to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")
import pandas as _pd
_pd.DataFrame(srows).to_csv(SUMMARY_CSV, index=False)
print(f"wrote {SUMMARY_CSV}")
